In [39]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/segment1_high_value_high_risk.csv
/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/segment2_high_value_low_risk.csv
/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/churn_prob_data.csv
/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/aggregate_churn.csv


### Data Step

In [40]:
import pandas as pd
import numpy as np

np.random.seed(101)

# Using the same 500 users from previous days
user_ids = np.arange(0, 500)

# Randomly assign 50% to Control, 50% to Treatment
variants = np.random.choice(['Control', 'Treatment'], size=500, p=[0.5, 0.5])

# Simulate post-test core events (Treatment gets a slight artificial bump to simulate a successful feature)
base_events = np.random.poisson(lam=15, size=500) 
treatment_effect = np.where(variants == 'Treatment', np.random.poisson(lam=3, size=500), 0)

post_test_events = base_events + treatment_effect

df_experiment = pd.DataFrame({
    'user_id': user_ids,
    'variant': variants,
    'post_test_core_events': post_test_events
})

In [41]:
seg1_df = pd.read_csv('/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/segment1_high_value_high_risk.csv')
seg2_df = pd.read_csv('/kaggle/input/datasets/sktechno710/aggregate-churn-data-made-from-collabspace-data/segment2_high_value_low_risk.csv')

In [42]:
seg1_df

,Unnamed: 0,user_id,subscription_tier,monthly_fee,gross_margin_pct,churn,projected_cltv
0,30,30,Pro,50,0.85,0.03,1416.666667
1,69,69,Enterprise,200,0.75,0.04,3750.000000
2,115,115,Pro,50,0.85,0.02,2125.000000
3,127,127,Pro,50,0.85,0.03,1416.666667
4,134,134,Enterprise,200,0.75,0.07,2142.857143
5,182,182,Enterprise,200,0.75,0.04,3750.000000
6,198,198,Pro,50,0.85,0.02,2125.000000
7,240,240,Enterprise,200,0.75,0.08,1875.000000
8,256,256,Enterprise,200,0.75,0.06,2500.000000
9,260,260,Pro,50,0.85,0.03,1416.666667


In [43]:
seg2_df

,Unnamed: 0,user_id,subscription_tier,monthly_fee,gross_margin_pct,churn,projected_cltv
0,1,1,Enterprise,200,0.75,0.00,15000.0
1,7,7,Pro,50,0.85,0.00,4250.0
2,8,8,Pro,50,0.85,0.01,4250.0
3,9,9,Pro,50,0.85,0.00,4250.0
4,11,11,Enterprise,200,0.75,0.00,15000.0
...,...,...,...,...,...,...,...
99,481,481,Pro,50,0.85,0.00,4250.0
100,488,488,Enterprise,200,0.75,0.00,15000.0
101,489,489,Pro,50,0.85,0.00,4250.0
102,491,491,Pro,50,0.85,0.00,4250.0


In [44]:
df_experiment

,user_id,variant,post_test_core_events
0,0,Treatment,22
1,1,Treatment,16
2,2,Control,24
3,3,Control,8
4,4,Treatment,14
...,...,...,...
495,495,Control,12
496,496,Treatment,19
497,497,Control,15
498,498,Treatment,14


In [45]:
set1 = set(seg1_df['user_id'])
set2 = set(seg2_df['user_id'])
search_space = set1.union(set2)
search_space

{1,
 7,
 8,
 9,
 11,
 12,
 20,
 25,
 30,
 34,
 45,
 53,
 55,
 62,
 67,
 69,
 73,
 74,
 75,
 76,
 81,
 86,
 92,
 107,
 112,
 113,
 115,
 121,
 127,
 134,
 137,
 140,
 146,
 159,
 161,
 162,
 170,
 176,
 178,
 182,
 183,
 185,
 186,
 196,
 197,
 198,
 200,
 203,
 207,
 212,
 214,
 217,
 219,
 226,
 228,
 230,
 238,
 240,
 241,
 245,
 248,
 256,
 257,
 260,
 261,
 265,
 266,
 267,
 270,
 272,
 276,
 277,
 278,
 279,
 280,
 292,
 314,
 324,
 327,
 351,
 355,
 358,
 367,
 373,
 378,
 381,
 382,
 384,
 385,
 393,
 396,
 399,
 401,
 403,
 408,
 411,
 412,
 414,
 419,
 421,
 427,
 431,
 432,
 435,
 437,
 441,
 445,
 446,
 450,
 462,
 471,
 473,
 475,
 480,
 481,
 488,
 489,
 491,
 498,
 499}

In [46]:
def targeted(row):
    if row['user_id'] in search_space: 
        return 1
    else: 
        return 0
df_experiment['target'] = df_experiment.apply(targeted,axis=1)

In [47]:
df_experiment

,user_id,variant,post_test_core_events,target
0,0,Treatment,22,0
1,1,Treatment,16,1
2,2,Control,24,0
3,3,Control,8,0
4,4,Treatment,14,0
...,...,...,...,...
495,495,Control,12,0
496,496,Treatment,19,0
497,497,Control,15,0
498,498,Treatment,14,1


In [48]:
df_treatment = df_experiment[df_experiment['target']==1]
df_control = df_experiment[df_experiment['target']==0]

In [49]:
df_treatment

,user_id,variant,post_test_core_events,target
1,1,Treatment,16,1
7,7,Treatment,17,1
8,8,Treatment,18,1
9,9,Control,17,1
11,11,Control,15,1
...,...,...,...,...
488,488,Control,17,1
489,489,Control,12,1
491,491,Control,12,1
498,498,Treatment,14,1


In [50]:
df_control

,user_id,variant,post_test_core_events,target
0,0,Treatment,22,0
2,2,Control,24,0
3,3,Control,8,0
4,4,Treatment,14,0
5,5,Treatment,21,0
...,...,...,...,...
493,493,Control,19,0
494,494,Treatment,22,0
495,495,Control,12,0
496,496,Treatment,19,0


In [51]:
mean1 = df_treatment['post_test_core_events'].mean()
std1 = df_treatment['post_test_core_events'].std()
mean2 = df_control['post_test_core_events'].mean()
std2 = df_control['post_test_core_events'].std()

In [52]:
print(f"The Difference in mean of treatment and control group is {mean1-mean2}")
print(f"The Difference in std of treatment and control group is {std1-std2}")

The Difference in mean of treatment and control group is -0.6469298245614041
The Difference in std of treatment and control group is -0.46821421824402254


### Calculating T Statistic and P value for validating new product Launch

In [53]:
!pip install scipy

In [54]:
from scipy import stats

t_stat, p_val = stats.ttest_ind(df_treatment['post_test_core_events'], 
                                df_control['post_test_core_events'], 
                                equal_var=False)

In [55]:
print(t_stat)
print(p_val)

-1.513880751361672
0.13148677412531243
